# LWCNN v3 — Full Evaluation Notebook
Loads `model_lwcnn_drone_binary.keras` from Drive and produces:
- Internal test set: confusion matrices at t=0.5 and t=0.3, classification reports, ROC curve
- External Svanström: confusion matrices, clip + window-level AUC
- Threshold sweep with plots
- Temperature scaling: before vs after comparison
- Computational benchmarks: parameter count, inference time, model size
- All thesis-ready figures saved to Drive

In [1]:
!pip -q install librosa scikit-learn pandas scipy matplotlib

In [2]:
import os, glob, zipfile, requests, time
import numpy as np
import pandas as pd
import librosa
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LinearSegmentedColormap
import scipy.optimize
import tensorflow as tf
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve
)
from google.colab import drive
drive.mount('/content/drive')

print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

Mounted at /content/drive
TF: 2.20.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# ── Config — matches LWCNN v3 training pipeline exactly ──────────────────────────────────────────────
SEED       = 42
rng        = np.random.default_rng(SEED)
tf.random.set_seed(SEED)

WIN_S      = 0.5
HOP_S      = 0.25
SR_TARGET  = 16000
N_MELS     = 64
N_FFT      = 512
HOP_LENGTH = 128
FMAX       = 8000
TOP_DB     = 80
AUG_PROB   = 0.7
SNR_MIN    = -5.0
SNR_MAX    = 20.0
BATCH      = 64

OUT_DIR    = '/content/drive/MyDrive/drone_audio_processed'
FIG_DIR    = os.path.join(OUT_DIR, 'figures', 'lwcnn')
os.makedirs(FIG_DIR, exist_ok=True)

MODEL_PATH = os.path.join(OUT_DIR, 'model_lwcnn_drone_binary.keras')
NPZ_PATH   = os.path.join(OUT_DIR, 'waveform_0.50s_hop0.25s_recordsplit.npz')

DEMAND_DIR         = '/content/DEMAND'
DEMAND_ZENODO_BASE = 'https://zenodo.org/records/1227121/files'
DEMAND_16K_ENVS    = [
    'DKITCHEN','DLIVING','DWASHING','NFIELD','NPARK','NRIVER',
    'OHALLWAY','OMEETING','OOFFICE','PCAFETER','PRESTO','PSTATION',
    'SPSQUARE','STRAFFIC','TBUS','TCAR','TMETRO',
]

EXT_ROOT    = '/content/external_drone_thesis'
EXT_ZIP_URL = 'https://codeload.github.com/DroneDetectionThesis/Drone-detection-dataset/zip/refs/heads/master'
THRESHOLDS  = np.round(np.arange(0.05, 0.96, 0.05), 2)

plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10})
CMAP = LinearSegmentedColormap.from_list('thesis_blue', ['#FFFFFF', '#2E74B5'], N=256)
print('Config loaded.')


Config loaded.


In [4]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def log_mel_db(y, sr=SR_TARGET):
    mel = librosa.feature.melspectrogram(
        y=y.astype(np.float32), sr=sr, n_mels=N_MELS,
        n_fft=N_FFT, hop_length=HOP_LENGTH, fmax=FMAX, power=2.0
    )
    return librosa.power_to_db(mel, ref=1.0, top_db=TOP_DB).astype(np.float32)

def mel_to_model_array(mel):
    mel  = mel[..., None].astype(np.float32)
    mean = np.mean(mel); std = np.std(mel)
    return ((mel - mean) / (std + 1e-6)).astype(np.float32)

def windows_all(y, sr, win_s, hop_s, rng_local):
    y   = np.asarray(y, dtype=np.float32)
    win = int(round(win_s * sr))
    hop = int(round(hop_s * sr))
    if len(y) <= win:
        out = np.zeros(win, dtype=np.float32)
        start = rng_local.integers(0, win - len(y) + 1)
        out[start:start + len(y)] = y
        return [out]
    return [y[s:s + win] for s in range(0, len(y) - win + 1, hop)]

def add_real_noise_snr(wave, noise_bank, snr_db, rng_local):
    if not noise_bank: return wave
    noise = noise_bank[rng_local.integers(0, len(noise_bank))]
    if len(noise) < len(wave):
        noise = np.tile(noise, int(np.ceil(len(wave)/len(noise))))
    start = rng_local.integers(0, len(noise) - len(wave) + 1)
    noise = noise[start:start + len(wave)]
    p_sig = np.mean(wave**2) + 1e-9
    p_noi = np.mean(noise**2) + 1e-9
    scale = np.sqrt(p_sig / (p_noi * 10**(snr_db/10)))
    return (wave + scale*noise).astype(np.float32)

def drone_metrics(y_true, y_prob, t):
    pred = (y_prob >= t).astype(int)
    cm   = confusion_matrix(y_true, pred)
    if cm.shape != (2,2): return 0., 0., 0.
    tp=cm[1,1]; fp=cm[0,1]; fn=cm[1,0]
    prec = tp/(tp+fp+1e-9); rec = tp/(tp+fn+1e-9)
    return prec, rec, 2*prec*rec/(prec+rec+1e-9)

def save_fig(fig, name):
    path = os.path.join(FIG_DIR, name)
    fig.savefig(path, dpi=200, bbox_inches='tight', facecolor='white')
    print(f'  Saved: {name}')

def plot_cm(cm, title, ax):
    thresh = cm.max() / 2.0
    ax.imshow(cm, interpolation='nearest', cmap=CMAP, vmin=0, vmax=cm.max())
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(['No Drone','Drone'], fontsize=9)
    ax.set_yticklabels(['No Drone','Drone'], fontsize=9)
    ax.set_xlabel('Predicted', fontsize=9, labelpad=8)
    ax.set_ylabel('True', fontsize=9, labelpad=18)
    ax.set_title(title, fontsize=9, fontweight='normal', pad=10)
    for row in range(2):
        row_total = cm[row].sum()
        for col in range(2):
            count = cm[row,col]
            pct   = count/row_total*100
            abbr  = [['TN','FP'],['FN','TP']][row][col]
            color = 'white' if count > thresh else '#1a1a1a'
            ax.text(col, row-0.12, abbr, ha='center', va='center',
                    fontsize=10, fontweight='bold', color=color)
            ax.text(col, row+0.12, f'{count:,}\n({pct:.1f}%)',
                    ha='center', va='center', fontsize=9, color=color)

print('Helpers defined.')

Helpers defined.


In [5]:
# ── Load DEMAND noise bank ────────────────────────────────────────────────────

os.makedirs(DEMAND_DIR, exist_ok=True)
demand_wavs = glob.glob(os.path.join(DEMAND_DIR,'**','*.wav'), recursive=True)

if len(demand_wavs) == 0:
    print('Downloading DEMAND...')
    for env in DEMAND_16K_ENVS:
        url = f'{DEMAND_ZENODO_BASE}/{env}_16k.zip?download=1'
        zip_path = f'/content/{env.lower()}.zip'
        print(f'  {env}...', end=' ', flush=True)
        r = requests.get(url, stream=True); r.raise_for_status()
        with open(zip_path, 'wb') as f:
            for chunk in r.iter_content(1024*1024): f.write(chunk)
        with zipfile.ZipFile(zip_path) as z: z.extractall(DEMAND_DIR)
        os.remove(zip_path)
        print('done')
    demand_wavs = glob.glob(os.path.join(DEMAND_DIR,'**','*.wav'), recursive=True)

noise_bank = []
seg_n = int(3.0 * SR_TARGET)
for p in rng.permutation(demand_wavs).tolist():
    try:
        y, sr = librosa.load(p, sr=None, mono=True)
        if sr != SR_TARGET: y = librosa.resample(y, orig_sr=sr, target_sr=SR_TARGET)
        for start in range(0, len(y)-seg_n+1, seg_n):
            seg = y[start:start+seg_n].astype(np.float32)
            if np.max(np.abs(seg)) > 1e-4: noise_bank.append(seg)
    except: pass

print(f'Noise bank: {len(noise_bank)} segments loaded.')

  DKITCHEN... done
  DLIVING... done
  DWASHING... done
  NFIELD... done
  NPARK... done
  NRIVER... done
  OHALLWAY... done
  OMEETING... done
  OOFFICE... done
  PCAFETER... done
  PRESTO... done
  PSTATION... done
  SPSQUARE... done
  STRAFFIC... done
  TBUS... done
  TCAR... 

HTTPError: 504 Server Error: Gateway Time-out for url: https://zenodo.org/records/1227121/files/TCAR_16k.zip?download=1

In [ ]:
# ── Load model and benchmark ──────────────────────────────────────────────────

print(f'Loading: {MODEL_PATH}')
model = tf.keras.models.load_model(MODEL_PATH, compile=False)

total_params     = model.count_params()
trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
model_size_mb    = os.path.getsize(MODEL_PATH) / (1024**2)

print(f'\n=== COMPUTATIONAL PROFILE ===')
print(f'Total parameters     : {total_params:,}')
print(f'Trainable parameters : {trainable_params:,}')
print(f'Model size on disk   : {model_size_mb:.2f} MB')
print(f'LWCNN params (ref.)  : {LWCNN_PARAMS:,}')
print(f'Parameter overhead   : +{total_params - LWCNN_PARAMS:,} ({(total_params/LWCNN_PARAMS - 1)*100:.1f}%)')
model.summary()

In [ ]:
# ── Load internal test set ────────────────────────────────────────────────────

print(f'Loading NPZ: {NPZ_PATH}')
npz = np.load(NPZ_PATH, allow_pickle=True)
print(f'Keys: {list(npz.keys())}')

X_test_wav = npz['X_test_wav'].astype(np.float32)
y_test     = npz['y_test_wav'].astype(int)
X_val_wav  = npz['X_val_wav'].astype(np.float32)
y_val      = npz['y_val_wav'].astype(int)
sr_used    = int(npz['sr']) if 'sr' in npz else SR_TARGET

print(f'Test: {X_test_wav.shape} | Val: {X_val_wav.shape}')

def build_mels(X_wav, augment=True, label=''):
    local_rng = np.random.default_rng(SEED)
    mels = []
    for i, w in enumerate(X_wav):
        if i % 10000 == 0: print(f'  {label} {i}/{len(X_wav)}')
        if augment and local_rng.random() < AUG_PROB:
            snr_db = local_rng.uniform(SNR_MIN, SNR_MAX)
            w = add_real_noise_snr(w, noise_bank, snr_db, local_rng)
        mels.append(mel_to_model_array(log_mel_db(w, sr_used)))
    return np.stack(mels).astype(np.float32)

print('Building test mels (with noise augmentation)...')
X_test = build_mels(X_test_wav, augment=True, label='test')
print('Building val mels (no augmentation)...')
X_val  = build_mels(X_val_wav,  augment=False, label='val')
print(f'Test mel shape: {X_test.shape}')

In [ ]:
# ── Inference time benchmark ──────────────────────────────────────────────────

_ = model.predict(X_test[:10], batch_size=BATCH, verbose=0)  # warm up

N_SINGLE = 100
t0 = time.perf_counter()
for _ in range(N_SINGLE):
    model.predict(X_test[:1], batch_size=1, verbose=0)
single_ms = (time.perf_counter() - t0) / N_SINGLE * 1000

t0 = time.perf_counter()
test_logits = model.predict(X_test, batch_size=BATCH, verbose=1).ravel()
batch_s = time.perf_counter() - t0
per_win_ms = batch_s / len(X_test) * 1000

test_probs = tf.sigmoid(test_logits).numpy()

print(f'\n=== INFERENCE TIMES ===')
print(f'Single window (mean {N_SINGLE} runs) : {single_ms:.2f} ms')
print(f'Full test set ({len(X_test):,} windows)  : {batch_s:.2f} s')
print(f'Per window (batched)                  : {per_win_ms:.3f} ms')
print(f'Real-time factor (0.5s window)        : {500/single_ms:.0f}x')

In [ ]:
# ── Internal test set evaluation ──────────────────────────────────────────────

int_auc = roc_auc_score(y_test, test_probs)
print(f'\n=== INTERNAL TEST SET ===')
print(f'ROC-AUC: {int_auc:.4f}')

for t, label in [(0.5,'Standard (t=0.5)'), (0.3,'Optimised (t=0.3)')]:
    pred = (test_probs >= t).astype(int)
    acc  = (pred == y_test).mean()
    cm   = confusion_matrix(y_test, pred)
    print(f'\n--- {label} ---')
    print(f'Accuracy: {acc:.4f}')
    print('Confusion matrix:')
    print(cm)
    print(classification_report(y_test, pred, digits=4,
                                target_names=['No Drone','Drone']))


In [ ]:
# ── Temperature scaling ───────────────────────────────────────────────────────

val_logits = model.predict(X_val, batch_size=BATCH, verbose=0).ravel()

def nll(log_T):
    T = np.exp(log_T[0])
    p = 1/(1+np.exp(-val_logits/T))
    eps = 1e-7
    return -np.mean(y_val*np.log(p+eps) + (1-y_val)*np.log(1-p+eps))

result = scipy.optimize.minimize(nll, x0=[0.0], method='L-BFGS-B')
T = float(np.exp(result.x[0]))
print(f'Temperature T = {T:.4f}')

test_probs_cal = tf.sigmoid(test_logits / T).numpy()
int_auc_cal    = roc_auc_score(y_test, test_probs_cal)

drone_mask   = y_test == 1
nodrone_mask = y_test == 0
print(f'\n=== CALIBRATION IMPACT (INTERNAL) ===')
print(f'AUC uncalibrated : {int_auc:.4f}')
print(f'AUC calibrated   : {int_auc_cal:.4f}')
print(f'Drone prob — uncal mean: {test_probs[drone_mask].mean():.3f} | cal: {test_probs_cal[drone_mask].mean():.3f}')
print(f'No-drone prob — uncal mean: {test_probs[nodrone_mask].mean():.3f} | cal: {test_probs_cal[nodrone_mask].mean():.3f}')

In [ ]:
# ── External Svanström evaluation ─────────────────────────────────────────────

os.makedirs(EXT_ROOT, exist_ok=True)
ext_zip = os.path.join(EXT_ROOT, 'repo.zip')
ext_dir = os.path.join(EXT_ROOT, 'repo')

if not os.path.exists(ext_zip):
    print('Downloading Svanström...')
    r = requests.get(EXT_ZIP_URL, stream=True); r.raise_for_status()
    with open(ext_zip, 'wb') as f:
        for chunk in r.iter_content(1024*1024): f.write(chunk)
    with zipfile.ZipFile(ext_zip) as z: z.extractall(ext_dir)

subdirs  = [p for p in glob.glob(os.path.join(ext_dir,'*')) if os.path.isdir(p)]
ext_repo = subdirs[0]
ext_wavs = sorted(glob.glob(os.path.join(ext_repo,'**','*.wav'), recursive=True))
print(f'External dataset: {len(ext_wavs)} clips')

local_rng = np.random.default_rng(SEED)
ext_logits_list, ext_ytrue = [], []

print('Running clip-level external evaluation...')
for p in ext_wavs:
    fn  = os.path.basename(p).lower()
    lab = 1 if 'drone' in fn else 0
    y, sr = librosa.load(p, sr=None, mono=True)
    if sr != SR_TARGET: y = librosa.resample(y, orig_sr=sr, target_sr=SR_TARGET)
    wins = windows_all(y, SR_TARGET, WIN_S, HOP_S, local_rng)
    X = []
    for w in wins:
        if local_rng.random() < AUG_PROB:
            snr_db = local_rng.uniform(SNR_MIN, SNR_MAX)
            w = add_real_noise_snr(w, noise_bank, snr_db, local_rng)
        X.append(mel_to_model_array(log_mel_db(w, SR_TARGET)))
    if not X: continue
    logits_c = model.predict(np.stack(X).astype(np.float32), batch_size=BATCH, verbose=0).ravel()
    ext_logits_list.append(logits_c.max())  # max logit = max-pool clip score
    ext_ytrue.append(lab)

ext_ytrue      = np.array(ext_ytrue, dtype=int)
ext_logits_arr = np.array(ext_logits_list, dtype=np.float32)
ext_probs      = tf.sigmoid(ext_logits_arr).numpy()
ext_probs_cal  = tf.sigmoid(ext_logits_arr / T).numpy()

ext_auc     = roc_auc_score(ext_ytrue, ext_probs)
ext_auc_cal = roc_auc_score(ext_ytrue, ext_probs_cal)
print(f'External clip AUC (uncalibrated): {ext_auc:.4f}')
print(f'External clip AUC (calibrated)  : {ext_auc_cal:.4f}')

ext_drone_mask = ext_ytrue == 1
for probs, label in [(ext_probs,'Uncal'), (ext_probs_cal,f'Cal T={T:.3f}')]:
    d = probs[ext_drone_mask]
    print(f'{label} — ext drone: mean={d.mean():.3f}, median={np.median(d):.3f}, min={d.min():.3f}, max={d.max():.3f}')

In [ ]:
# ── Window-level external evaluation ─────────────────────────────────────────

print('Running window-level external evaluation...')
local_rng = np.random.default_rng(SEED)
ext_win_probs, ext_win_labels = [], []

for p in ext_wavs:
    fn  = os.path.basename(p).lower()
    lab = 1 if 'drone' in fn else 0
    y, sr = librosa.load(p, sr=None, mono=True)
    if sr != SR_TARGET: y = librosa.resample(y, orig_sr=sr, target_sr=SR_TARGET)
    wins = windows_all(y, SR_TARGET, WIN_S, HOP_S, local_rng)
    X = []
    for w in wins:
        if local_rng.random() < AUG_PROB:
            snr_db = local_rng.uniform(SNR_MIN, SNR_MAX)
            w = add_real_noise_snr(w, noise_bank, snr_db, local_rng)
        X.append(mel_to_model_array(log_mel_db(w, SR_TARGET)))
    if not X: continue
    logits_c = model.predict(np.stack(X).astype(np.float32), batch_size=BATCH, verbose=0).ravel()
    ext_win_probs.extend(tf.sigmoid(logits_c).numpy().tolist())
    ext_win_labels.extend([lab]*len(logits_c))

ext_win_probs  = np.array(ext_win_probs,  dtype=np.float32)
ext_win_labels = np.array(ext_win_labels, dtype=int)
win_auc = roc_auc_score(ext_win_labels, ext_win_probs)
print(f'Window-level external AUC: {win_auc:.4f} ({len(ext_win_probs)} windows)')

In [ ]:
# ── Threshold sweep ───────────────────────────────────────────────────────────

def run_sweep(int_pr, ext_pr, label):
    rows = []
    for t in THRESHOLDS:
        _, _, if1  = drone_metrics(y_test,   int_pr, t)
        _, er, ef1 = drone_metrics(ext_ytrue, ext_pr, t)
        rows.append(dict(threshold=t, int_drone_f1=round(if1,4),
                         ext_drone_recall=round(er,4), ext_drone_f1=round(ef1,4),
                         combined=round(2*if1*er/(if1+er+1e-9),4)))
    df   = pd.DataFrame(rows)
    best = df.loc[df['combined'].idxmax()]
    print(f'\n=== SWEEP ({label}) ===')
    print(df.to_string(index=False))
    print(f'Optimal: t={best["threshold"]} | F1={best["int_drone_f1"]} | Ext recall={best["ext_drone_recall"]} | Combined={best["combined"]}')
    return df, best

df_uncal, best_uncal = run_sweep(test_probs,     ext_probs,     'Uncalibrated')
df_cal,   best_cal   = run_sweep(test_probs_cal, ext_probs_cal, f'Calibrated T={T:.3f}')

OPT_T_UNCAL = float(best_uncal['threshold'])
OPT_T_CAL   = float(best_cal['threshold'])

print(f'\n=== TEMPERATURE SCALING IMPACT ===')
print(f'T                               : {T:.4f}')
print(f'Optimal threshold (uncal)       : {OPT_T_UNCAL}')
print(f'Optimal threshold (cal)         : {OPT_T_CAL}')
print(f'Combined score (uncal)          : {best_uncal["combined"]}')
print(f'Combined score (cal)            : {best_cal["combined"]}')
print(f'Ext. recall (uncal)             : {best_uncal["ext_drone_recall"]}')
print(f'Ext. recall (cal)               : {best_cal["ext_drone_recall"]}')

In [ ]:
# ── FIGURE 1: Confusion matrices — internal t=0.5 and t=0.3 ─────────────────

cm_05 = confusion_matrix(y_test, (test_probs >= 0.5).astype(int))
cm_03 = confusion_matrix(y_test, (test_probs >= 0.3).astype(int))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.subplots_adjust(wspace=0.55)
plot_cm(cm_05, 'LWCNN Internal Test Set\n(threshold = 0.5)',  axes[0])
plot_cm(cm_03, 'LWCNN Internal Test Set\n(threshold = 0.3)', axes[1])
for ax in axes:
    from mpl_toolkits.axes_grid1 import make_axes_locatable
    divider = make_axes_locatable(ax)
    cax = divider.append_axes('right', size='5%', pad=0.1)
    plt.colorbar(ax.images[0], cax=cax, label='Count')
save_fig(fig, 'fig_lwcnn_cm_internal.png')
plt.show()


In [ ]:
# ── FIGURE 2: Confusion matrices — external t=0.5 and t=0.3 ─────────────────

cm_ext_05 = confusion_matrix(ext_ytrue, (ext_probs >= 0.5).astype(int))
cm_ext_03 = confusion_matrix(ext_ytrue, (ext_probs >= 0.3).astype(int))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.subplots_adjust(wspace=0.55)
plot_cm(cm_ext_05, 'LWCNN External Test Set\n(threshold = 0.5)', axes[0])
plot_cm(cm_ext_03, 'LWCNN External Test Set\n(threshold = 0.3)', axes[1])
for ax in axes:
    from mpl_toolkits.axes_grid1 import make_axes_locatable
    divider = make_axes_locatable(ax)
    cax = divider.append_axes('right', size='5%', pad=0.1)
    plt.colorbar(ax.images[0], cax=cax, label='Count')
save_fig(fig, 'fig_lwcnn_cm_external.png')
plt.show()


In [ ]:
# ── FIGURE 3: Threshold sweep — uncal vs cal ──────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(df_uncal['threshold'], df_uncal['int_drone_f1'],     label='Int. drone F1',   lw=2, marker='o', ms=4)
ax.plot(df_uncal['threshold'], df_uncal['ext_drone_recall'], label='Ext. drone recall',lw=2, marker='s', ms=4)
ax.plot(df_uncal['threshold'], df_uncal['ext_drone_f1'],     label='Ext. drone F1',   lw=2, marker='^', ms=4)
ax.plot(df_uncal['threshold'], df_uncal['combined'],         label='Combined (HM)',    lw=2, ls='--', marker='D', ms=4)
ax.axvline(OPT_T_UNCAL, color='red', ls=':', lw=1.5, label=f'Optimal t={OPT_T_UNCAL}')
ax.set_xlabel('Threshold'); ax.set_ylabel('Score')
ax.set_title('(a) CNN-Transformer: Threshold vs Metrics', loc='left')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(df_uncal['threshold'], df_uncal['combined'], lw=2.5, color='#185FA5', marker='D', ms=4, label='Uncalibrated')
ax.plot(df_cal['threshold'],   df_cal['combined'],   lw=2.5, color='coral',   marker='o', ms=4, ls='--', label=f'Calibrated (T={T:.3f})')
ax.axvline(OPT_T_UNCAL, color='#185FA5', ls=':', lw=1.5)
ax.axvline(OPT_T_CAL,   color='coral',   ls=':', lw=1.5)
ax.set_xlabel('Threshold'); ax.set_ylabel('Combined Score')
ax.set_title('(b) Combined Score: Calibrated vs Uncalibrated', loc='left')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

fig.suptitle('LWCNN — Decision Threshold Optimisation', fontsize=12, y=1.01)
plt.tight_layout()
save_fig(fig, 'fig_lwcnn_threshold_sweep.png')
plt.show()

In [ ]:
# ── FIGURE 4: Probability histograms — calibration before/after ───────────────

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
bins = np.linspace(0, 1, 25)

# Top row: external drone clips
ax = axes[0, 0]
ax.hist(ext_probs[ext_drone_mask],     bins=bins, alpha=0.7, color='coral',     density=True, label='Uncalibrated')
ax.hist(ext_probs_cal[ext_drone_mask], bins=bins, alpha=0.7, color='steelblue', density=True, label='Calibrated')
ax.axvline(OPT_T_UNCAL, color='coral',     ls='--', lw=1.5, label=f'Uncal. t={OPT_T_UNCAL}')
ax.axvline(OPT_T_CAL,   color='steelblue', ls='--', lw=1.5, label=f'Cal. t={OPT_T_CAL}')
ax.set_xlabel('Probability'); ax.set_ylabel('Density')
ax.set_title('(a) External Drone Clip Probabilities', loc='left')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Top right: external no-drone clips
ax = axes[0, 1]
ax.hist(ext_probs[ext_ytrue==0],     bins=bins, alpha=0.7, color='coral',     density=True, label='Uncalibrated')
ax.hist(ext_probs_cal[ext_ytrue==0], bins=bins, alpha=0.7, color='steelblue', density=True, label='Calibrated')
ax.set_xlabel('Probability'); ax.set_ylabel('Density')
ax.set_title('(b) External No-Drone Clip Probabilities', loc='left')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Bottom left: internal drone vs external drone (calibrated)
ax = axes[1, 0]
ax.hist(test_probs_cal[drone_mask],    bins=bins, alpha=0.6, color='#2E7D32', density=True, label='Internal drone (cal.)')
ax.hist(ext_probs_cal[ext_drone_mask], bins=bins, alpha=0.6, color='steelblue', density=True, label='External drone (cal.)')
ax.axvline(OPT_T_CAL, color='red', ls='--', lw=1.5, label=f'Optimal t={OPT_T_CAL}')
ax.set_xlabel('Probability'); ax.set_ylabel('Density')
ax.set_title('(c) Internal vs External Drone (Calibrated)', loc='left')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Bottom right: internal drone vs no-drone (calibrated)
ax = axes[1, 1]
ax.hist(test_probs_cal[drone_mask],   bins=bins, alpha=0.6, color='#2E7D32', density=True, label='Internal drone (cal.)')
ax.hist(test_probs_cal[nodrone_mask], bins=bins, alpha=0.6, color='#C62828', density=True, label='Internal no-drone (cal.)')
ax.axvline(0.5, color='black', ls='--', lw=1.5, label='t=0.5')
ax.set_xlabel('Probability'); ax.set_ylabel('Density')
ax.set_title('(d) Internal Class Separation (Calibrated)', loc='left')
ax.legend(fontsize=8); ax.grid(alpha=0.3)

fig.suptitle(f'LWCNN — Temperature Scaling Effect (T={T:.4f})', fontsize=12, y=1.01)
plt.tight_layout()
save_fig(fig, 'fig_lwcnn_calibration.png')
plt.show()

In [ ]:
# ── FIGURE 5: ROC curves — internal and external ──────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

fpr_i, tpr_i, _ = roc_curve(y_test, test_probs)
axes[0].plot(fpr_i, tpr_i, lw=2, color='#185FA5', label=f'CNN-Transformer (AUC={int_auc:.4f})')
axes[0].plot([0,1],[0,1],'k--',alpha=0.3)
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('(a) ROC Curve — Internal Test Set', loc='left')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

fpr_e, tpr_e, _   = roc_curve(ext_ytrue, ext_probs)
fpr_ec, tpr_ec, _ = roc_curve(ext_ytrue, ext_probs_cal)
axes[1].plot(fpr_e,  tpr_e,  lw=2, color='#185FA5', label=f'Uncalibrated (AUC={ext_auc:.4f})')
axes[1].plot(fpr_ec, tpr_ec, lw=2, color='coral',   ls='--', label=f'Calibrated (AUC={ext_auc_cal:.4f})')
axes[1].plot([0,1],[0,1],'k--',alpha=0.3)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('(b) ROC Curve — External Svanström', loc='left')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

fig.suptitle('LWCNN — ROC Curves', fontsize=12, y=1.01)
plt.tight_layout()
save_fig(fig, 'fig_lwcnn_roc.png')
plt.show()

In [ ]:
# ── Final summary ─────────────────────────────────────────────────────────────

print('\n' + '='*60)
print('LWCNN v3 — FULL RESULTS SUMMARY')
print('='*60)
print(f'Parameters           : {total_params:,}')
print(f'Model size           : {model_size_mb:.2f} MB')
print(f'Single window latency: {single_ms:.2f} ms')
print(f'Real-time factor     : {500/single_ms:.0f}x')
print()
print(f'Internal AUC         : {int_auc:.4f}')
print(f'Internal accuracy    : {((test_probs>=0.5).astype(int)==y_test).mean():.4f} (t=0.5)')
print(f'Internal drone F1    : {drone_metrics(y_test,test_probs,0.5)[2]:.4f} (t=0.5)')
print()
print(f'External clip AUC    : {ext_auc:.4f}')
print(f'External window AUC  : {win_auc:.4f}')
print(f'Temperature T        : {T:.4f}')
print(f'Optimal threshold    : {OPT_T_UNCAL} (uncal.) | {OPT_T_CAL} (cal.)')
print(f'Combined score       : {best_uncal["combined"]} (uncal.) | {best_cal["combined"]} (cal.)')
print(f'Ext. recall at opt t : {best_uncal["ext_drone_recall"]} (uncal.) | {best_cal["ext_drone_recall"]} (cal.)')
print(f'\nAll figures saved to: {FIG_DIR}')